<a href="https://colab.research.google.com/github/louisnguyen-eep/AgenticAIforBusiness118S/blob/dev/orderStatus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install anthropic langgraph langchain-core -q

import re
from datetime import datetime
from typing import Annotated
from typing_extensions import TypedDict

import anthropic
from google.colab import userdata

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# ── Claude Client ─────────────────────────────────────────────────────────────
api_key = userdata.get('ANTHROPIC_API_KEY')
client  = anthropic.Anthropic(api_key=api_key)
MODEL   = "claude-sonnet-4-5"

# ── Orders Database ───────────────────────────────────────────────────────────
ORDERS_DB = {
    "ORD-1001": {"status": "Shipped",      "item": "NexaPad Ultra",    "carrier": "FedEx", "tracking": "FX9284710234",           "estimated_date": "April 1, 2026"},
    "ORD-1002": {"status": "Processing",   "item": "VisionWatch Pro",  "carrier": "UPS",   "tracking": None,                     "estimated_date": "April 4, 2026"},
    "ORD-1003": {"status": "Delivered",    "item": "SoundDrop ANC",    "carrier": "USPS",  "tracking": "9400111899223456789012", "estimated_date": "March 25, 2026"},
    "ORD-1004": {"status": "Cancelled",    "item": "NexaLink Router",  "carrier": "N/A",   "tracking": None,                     "estimated_date": "N/A"},
    "ORD-1005": {"status": "Out for Delivery", "item": "NexaCam 4K",   "carrier": "FedEx", "tracking": "FX1122334455",           "estimated_date": "March 29, 2026"},
    "ORD-1006": {"status": "Processing",   "item": "ChargePad Trio",   "carrier": "UPS",   "tracking": None,                     "estimated_date": "April 3, 2026"},
    "ORD-1007": {"status": "Shipped",      "item": "DeskHub Pro",      "carrier": "USPS",  "tracking": "9400111899223456789099", "estimated_date": "April 2, 2026"},
    "ORD-1008": {"status": "Delivered",    "item": "NexaPad Lite",     "carrier": "FedEx", "tracking": "FX9988776655",           "estimated_date": "March 22, 2026"},
    "ORD-1009": {"status": "Shipped",      "item": "SoundDrop Go",     "carrier": "UPS",   "tracking": "1Z9999999999999999",     "estimated_date": "April 1, 2026"},
    "ORD-1010": {"status": "Processing",   "item": "VisionWatch SE",   "carrier": "FedEx", "tracking": None,                     "estimated_date": "April 5, 2026"},
}

# ── System Prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
You are Alex, a friendly customer support agent for NexaStore, a premium online tech retailer.
Your only job in this session is to help customers check their order status and shipping information.

BEHAVIOUR RULES:
- Be warm, concise, and professional.
- Use the exact order details provided in SYSTEM NOTEs — never invent information.
- If no order number is provided, politely ask for it (format: ORD-XXXX).
- If an order is not found, tell the customer and ask them to double-check.
- For delivered orders, confirm delivery and ask if everything arrived in good condition.
- For cancelled orders, empathise and direct them to support@nexastore.com if they need help.

COMPANY DETAILS:
- Support email : support@nexastore.com
- Support hours : Monday–Friday, 9 AM – 6 PM EST
- Website       : nexastore.com
""".strip()

# ── Claude Intent Classifier ──────────────────────────────────────────────────
def detect_intent(user_message: str) -> str:
    response = client.messages.create(
        model=MODEL,
        max_tokens=20,
        system="""Classify the customer message into exactly one of these intents:
check_order, no_order_number, general

Reply with only the intent label, nothing else.""",
        messages=[{"role": "user", "content": user_message}],
    )
    return response.content[0].text.strip().lower()

# ── Order Lookup ──────────────────────────────────────────────────────────────
def lookup_order(message: str) -> str:
    match = re.search(r'ORD-\d+', message, re.IGNORECASE)
    if not match:
        return "[SYSTEM NOTE: No order number found. Ask the customer for their order number (format: ORD-XXXX).]"

    order_id = match.group().upper()
    if order_id not in ORDERS_DB:
        return f"[SYSTEM NOTE: Order {order_id} was not found in the database. Tell the customer and ask them to double-check.]"

    o        = ORDERS_DB[order_id]
    tracking = f"Tracking number: {o['tracking']}" if o['tracking'] else "Tracking number not yet assigned."
    return (
        f"[SYSTEM NOTE - Order details for {order_id}:\n"
        f"  Item: {o['item']} | Status: {o['status']} | Carrier: {o['carrier']}\n"
        f"  Estimated Delivery: {o['estimated_date']} | {tracking}\n"
        f"Use these exact details in your reply.]"
    )

# ── LangGraph State ───────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# ── Claude Node ───────────────────────────────────────────────────────────────
def claude_node(state: AgentState) -> dict:
    claude_messages = []
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            claude_messages.append({"role": "user",      "content": msg.content})
        elif isinstance(msg, AIMessage):
            claude_messages.append({"role": "assistant", "content": msg.content})

    response = client.messages.create(
        model=MODEL,
        max_tokens=512,
        system=SYSTEM_PROMPT,
        messages=claude_messages,
    )
    reply = response.content[0].text.strip()
    return {"messages": [AIMessage(content=reply)]}

# ── Build Graph ───────────────────────────────────────────────────────────────
def build_graph() -> StateGraph:
    memory  = MemorySaver()
    builder = StateGraph(AgentState)
    builder.add_node("alex", claude_node)
    builder.add_edge(START, "alex")
    builder.add_edge("alex", END)
    return builder.compile(checkpointer=memory)

GRAPH = build_graph()

def invoke_graph(thread_id: str, human_content: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = GRAPH.invoke(
        {"messages": [HumanMessage(content=human_content)]},
        config=config,
    )
    return result["messages"][-1].content

# ── Main Chat Session ─────────────────────────────────────────────────────────
def run_chat_session():
    thread_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    print("=" * 60)
    print("  NexaStore — Order Status Agent")
    print(f"  Session ID (MemorySaver thread): {thread_id}")
    print("=" * 60)
    print("  Available orders: ORD-1001 through ORD-1010")
    print("  Type your message and press Enter. Type 'done' to exit.")
    print("-" * 60)

    # Greeting
    greeting = invoke_graph(thread_id, "Greet the customer warmly and let them know you can help check their order status.")
    print(f"\n  Alex: {greeting}\n")

    # Conversation loop
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in ("done", "quit", "exit", "bye"):
            break

        intent = detect_intent(user_input)
        print(f"  [Intent: {intent}]")

        note      = lookup_order(user_input)
        augmented = f"{user_input}\n\n{note}"

        reply = invoke_graph(thread_id, augmented)
        print(f"\n  Alex: {reply}\n")
        print("-" * 60)

    # Closing
    closing = invoke_graph(thread_id, "The customer is leaving. Give a warm one-sentence goodbye.")
    print(f"\n  Alex: {closing}\n")
    print("=" * 60)

    snapshot  = GRAPH.get_state({"configurable": {"thread_id": thread_id}})
    msg_count = len(snapshot.values["messages"])
    print(f"\n  [MemorySaver] {msg_count} messages checkpointed for thread '{thread_id}'")

# ── Run ───────────────────────────────────────────────────────────────────────
run_chat_session()

  NexaStore — Order Status Agent
  Session ID (MemorySaver thread): 20260330_002732
  Type your message and press Enter. Type 'done' to exit.
------------------------------------------------------------

  Alex: Hello! Welcome to NexaStore! 👋

I'm Alex, and I'm here to help you check on your order status today. Whether you're waiting on a new gadget or just want to know where your package is, I've got you covered!

To get started, could you please provide me with your order number? It should be in the format **ORD-XXXX** (you can find it in your confirmation email).

What's your order number? 😊

You: ORD-1001

  Alex: Great news! I've found your order. 🎉

**Order Number:** ORD-1001  
**Item:** AlphaBook Pro Laptop  
**Status:** Shipped ✅  
**Estimated Delivery:** March 21, 2026  
**Carrier:** FedEx  
**Tracking Number:** FX9284710234

Your AlphaBook Pro Laptop is on its way to you! You can track your package in real-time using the FedEx tracking number above on the FedEx website.

Is t